In [ ]:
# =============================================================================
# Complete Adaptive Framework for Mobile Money Fraud Detection
# Dataset: /kaggle/input/datasets/moonknightmarvel/paysim
# =============================================================================

# Auto‑install required library (if not present)
import sys, subprocess, importlib
try:
    importlib.import_module('river')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'river', '-q'])
    print("river installed successfully")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import glob
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score, fbeta_score, precision_recall_curve
from river.drift import ADWIN
import joblib

# Suppress LightGBM logs – register a dummy logger that satisfies the API
class DummyLogger:
    def info(self, msg): pass
    def warning(self, msg): pass

lgb.register_logger(DummyLogger())

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# ------------------------------ 1. Load Data ------------------------------
DATA_PATH = '../input/datasets/moonknightmarvel/paysim/'
csv_files = glob.glob(DATA_PATH + '*.csv')
if not csv_files:
    raise FileNotFoundError(f"No CSV file found in {DATA_PATH}")
print(f"Found CSV: {csv_files[0]}")
df = pd.read_csv(csv_files[0])

print(f"Shape: {df.shape} | Columns: {df.columns.tolist()}")
print("Fraud distribution:")
print(df['isFraud'].value_counts(normalize=True).map('{:.4%}'.format))

# ------------------------------ 2. Preprocessing & Feature Engineering ------------------------------
df['timestamp'] = pd.to_datetime('2020-01-01') + pd.to_timedelta(df['step'], unit='h')

type_dummies = pd.get_dummies(df['type'], prefix='type')
df = pd.concat([df, type_dummies], axis=1)

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_night'] = df['hour'].apply(lambda x: 1 if (x >= 22 or x < 6) else 0)
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

df['amount_log'] = np.log1p(df['amount'])
df['oldbalanceOrg_log'] = np.log1p(df['oldbalanceOrg'])
df['newbalanceOrig_log'] = np.log1p(df['newbalanceOrig'])
df['oldbalanceDest_log'] = np.log1p(df['oldbalanceDest'])
df['newbalanceDest_log'] = np.log1p(df['newbalanceDest'])

df['error_orig'] = df['newbalanceOrig'] - (df['oldbalanceOrg'] - df['amount'])
df['error_dest'] = df['newbalanceDest'] - (df['oldbalanceDest'] + df['amount'])
df['error_orig_abs'] = np.abs(df['error_orig'])
df['error_dest_abs'] = np.abs(df['error_dest'])

df['orig_zero_before'] = (df['oldbalanceOrg'] == 0).astype(int)
df['dest_zero_before'] = (df['oldbalanceDest'] == 0).astype(int)

# Expanding-window customer features (no lookahead)
def add_customer_features(data):
    data = data.sort_values('timestamp').reset_index(drop=True)
    g_orig = data.groupby('nameOrig')
    data['orig_tx_count'] = g_orig.cumcount()
    data['orig_cum_amount'] = g_orig['amount'].cumsum()
    data['orig_avg_amount'] = data['orig_cum_amount'] / (data['orig_tx_count'] + 1)
    data['orig_max_amount'] = g_orig['amount'].cummax()

    g_dest = data.groupby('nameDest')
    data['dest_tx_count'] = g_dest.cumcount()
    data['dest_cum_amount'] = g_dest['amount'].cumsum()
    data['dest_avg_amount'] = data['dest_cum_amount'] / (data['dest_tx_count'] + 1)
    data['dest_max_amount'] = g_dest['amount'].cummax()

    data['orig_prev_timestamp'] = g_orig['timestamp'].shift(1)
    data['orig_time_since_last'] = (data['timestamp'] - data['orig_prev_timestamp']).dt.total_seconds() / 3600
    data['orig_time_since_last'].fillna(0, inplace=True)
    return data

df = add_customer_features(df)

# Expanding target encoding (high-cardinality columns)
def expanding_target_encoding(data, cat_col, target_col='isFraud', alpha=100):
    data = data.sort_values('timestamp')
    global_mean = data[target_col].mean()
    agg = data.groupby(cat_col)[target_col].expanding().agg(['sum', 'count'])
    agg = agg.reset_index(level=0, drop=True)  # align with original index
    data['te_sum'] = agg['sum']
    data['te_count'] = agg['count']
    data['te_sum'] = data['te_sum'].fillna(0)
    data['te_count'] = data['te_count'].fillna(0)
    data[f'{cat_col}_te'] = (data['te_sum'] + alpha * global_mean) / (data['te_count'] + alpha)
    data.drop(['te_sum', 'te_count'], axis=1, inplace=True)
    return data

df = expanding_target_encoding(df, 'nameOrig')
df = expanding_target_encoding(df, 'nameDest')

drop_cols = ['step', 'type', 'nameOrig', 'nameDest', 'isFlaggedFraud',
             'timestamp', 'orig_prev_timestamp']
X_cols = [c for c in df.columns if c not in drop_cols + ['isFraud']]
print(f"Number of features: {len(X_cols)}")

# ------------------------------ 3. Adaptive Framework Setup ------------------------------
df['date'] = df['timestamp'].dt.date
weeks = sorted(df['date'].unique())
train_start_date = weeks[0]
train_end_date = train_start_date + timedelta(days=7)

drift_detector = ADWIN(delta=0.002)
performance_log = []

def prepare_data(data, X_cols):
    X = data[X_cols].copy().fillna(-999)
    y = data['isFraud'].copy()
    return X, y

def train_lightgbm(X_train, y_train, X_val, y_val):
    pos = y_train.sum()
    neg = len(y_train) - pos
    scale_pos_weight = neg / (pos + 1)
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'scale_pos_weight': scale_pos_weight,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'min_data_in_leaf': 50,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbosity': -1,
        'seed': 42
    }
    model = lgb.train(params,
                      dtrain,
                      num_boost_round=500,
                      valid_sets=[dval],
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    return model

def find_best_threshold(model, X_val, y_val, beta=2):
    probs = model.predict(X_val)
    prec, rec, thresh = precision_recall_curve(y_val, probs)
    fbeta = (1 + beta**2) * (prec * rec) / (beta**2 * prec + rec + 1e-10)
    best_idx = np.argmax(fbeta)
    return thresh[best_idx], fbeta[best_idx]

def evaluate_model(model, X_test, y_test, threshold=0.5):
    probs = model.predict(X_test)
    preds = (probs >= threshold).astype(int)
    f2 = fbeta_score(y_test, preds, beta=2)
    auc_roc = roc_auc_score(y_test, probs)
    report = classification_report(y_test, preds, output_dict=True)
    return {
        'f2_score': f2,
        'auc_roc': auc_roc,
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'threshold': threshold
    }

# ------------------------------ 4. Initial Training & Adaptive Loop ------------------------------
mask_train = df['date'] < train_end_date
df_train = df[mask_train]
df_rest = df[~mask_train]

split_date = df_train['date'].max() - timedelta(days=1)
df_init_train = df_train[df_train['date'] < split_date]
df_init_val = df_train[df_train['date'] >= split_date]

X_init_train, y_init_train = prepare_data(df_init_train, X_cols)
X_init_val, y_init_val = prepare_data(df_init_val, X_cols)

print("Initial training...")
model = train_lightgbm(X_init_train, y_init_train, X_init_val, y_init_val)
threshold, _ = find_best_threshold(model, X_init_val, y_init_val, beta=2)
print(f"Initial threshold: {threshold:.4f}")

metrics = evaluate_model(model, X_init_val, y_init_val, threshold)
print(f"Initial Val - F2: {metrics['f2_score']:.4f}, AUC: {metrics['auc_roc']:.4f}, "
      f"Recall: {metrics['recall']:.4f}, Precision: {metrics['precision']:.4f}")

current_model = model
current_threshold = threshold
all_predictions = []
all_labels = []

remaining_weeks = [d for d in weeks if d >= train_end_date]
week_groups = pd.Series(remaining_weeks).groupby(
    pd.to_datetime(remaining_weeks).to_period('W')).apply(list).tolist()

for week_dates in week_groups:
    mask_week = df['date'].isin(week_dates)
    df_week = df[mask_week].copy()
    if len(df_week) == 0:
        continue

    X_week, y_week = prepare_data(df_week, X_cols)
    probs_week = current_model.predict(X_week)
    preds_week = (probs_week >= current_threshold).astype(int)

    all_predictions.extend(preds_week)
    all_labels.extend(y_week)

    errors = (preds_week != y_week).astype(int)
    for err in errors:
        drift_detector.update(err)

    week_metrics = evaluate_model(current_model, X_week, y_week, current_threshold)
    performance_log.append({
        'week_start': week_dates[0],
        'n_samples': len(df_week),
        **week_metrics
    })

    retrain = False
    if drift_detector.drift_detected:
        print(f"Drift detected at {week_dates[0]}! Retraining...")
        retrain = True
        drift_detector.reset()

    if len(performance_log) % 4 == 0:
        print(f"Scheduled retrain at {week_dates[0]}")
        retrain = True

    if retrain:
        retrain_data = df[df['date'] < week_dates[0]]
        retrain_dates_sorted = sorted(retrain_data['date'].unique())
        split_idx = int(len(retrain_dates_sorted) * 0.8)
        split_date_retrain = retrain_dates_sorted[split_idx]
        df_retrain_train = retrain_data[retrain_data['date'] < split_date_retrain]
        df_retrain_val = retrain_data[retrain_data['date'] >= split_date_retrain]

        X_rt, y_rt = prepare_data(df_retrain_train, X_cols)
        X_rv, y_rv = prepare_data(df_retrain_val, X_cols)

        current_model = train_lightgbm(X_rt, y_rt, X_rv, y_rv)
        current_threshold, _ = find_best_threshold(current_model, X_rv, y_rv, beta=2)
        print(f"Retrained. New threshold: {current_threshold:.4f}")

# ------------------------------ 5. Final Evaluation & Visualisation ------------------------------
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

print("\nFinal Adaptive Model Performance on All Test Data:")
print(classification_report(all_labels, all_predictions, digits=4))

f2 = fbeta_score(all_labels, all_predictions, beta=2)
roc_auc = roc_auc_score(all_labels, all_predictions)
print(f"Overall F2-score: {f2:.4f}")
print(f"Overall AUC (binary preds): {roc_auc:.4f}")

perf_df = pd.DataFrame(performance_log)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(perf_df['week_start'], perf_df['f2_score'], marker='o')
axes[0,0].set_title('Weekly F2-score')
axes[0,0].tick_params(axis='x', rotation=45)

axes[0,1].plot(perf_df['week_start'], perf_df['auc_roc'], marker='o', color='orange')
axes[0,1].set_title('Weekly AUC-ROC')
axes[0,1].tick_params(axis='x', rotation=45)

axes[1,0].plot(perf_df['week_start'], perf_df['precision'], marker='o', color='green')
axes[1,0].plot(perf_df['week_start'], perf_df['recall'], marker='o', color='red')
axes[1,0].legend(['Precision', 'Recall'])
axes[1,0].set_title('Precision vs Recall')
axes[1,0].tick_params(axis='x', rotation=45)

axes[1,1].bar(perf_df['week_start'], perf_df['n_samples'], alpha=0.7)
axes[1,1].set_title('Weekly Volume')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

lgb.plot_importance(current_model, max_num_features=20, figsize=(10, 8))
plt.title('Feature Importance (Latest Model)')
plt.show()

# Optional: save model
joblib.dump({'model': current_model, 'threshold': current_threshold, 'X_cols': X_cols},
            'adaptive_fraud_model.pkl')
print("Model saved as adaptive_fraud_model.pkl")

Found CSV: ../input/datasets/moonknightmarvel/paysim/paysim.csv
Shape: (6362620, 11) | Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
Fraud distribution:
isFraud
0    99.8709%
1     0.1291%
Name: proportion, dtype: object
